### Overview

**Limitations**
1. `ai_query()` does not currently emit OTEL traces
2. AI Gateway inference tables do not support `ai_query()` as of now (no OTEL) - this is on the roadmap for 2026
3. MLFlow traces do not support `ai_query()` (same reason)
4. `QUERY_TAGS` can be set and used to trace cost more effectively, but they can only be used on SQL Warehouse as of today

**Approach**
Two options are outlined, either version prompt and model in a Delta Table or hardcode into a Unity Catalog registered function and version in GitHub as code.

When `ai_query()` is called, store the results in a Delta Table, this is your tracing. Use MLFlow to log metrics.






### Input Data

In [0]:
from pyspark.sql import Row

fake_data = [
    Row(review="Absolutely phenomenal. The acting was superb and the storyline kept me on the edge of my seat the entire time.", human_label_movie_good_bad=True),
    Row(review="A complete waste of two hours. The plot made no sense and the dialogue was painfully awkward.", human_label_movie_good_bad=False),
    Row(review="Visually stunning with a heartfelt story. I laughed, I cried — one of the best films this year.", human_label_movie_good_bad=True),
    Row(review="The pacing was unbearably slow and the characters were one-dimensional. I nearly walked out.", human_label_movie_good_bad=False),
    Row(review="A fun, lighthearted movie with great chemistry between the leads. Perfect for a weekend watch.", human_label_movie_good_bad=True),
    Row(review="Terrible CGI and a script that feels like it was written in ten minutes. Deeply disappointing.", human_label_movie_good_bad=False),
    Row(review="An instant classic. The director masterfully blends humor and drama into a truly memorable experience.", human_label_movie_good_bad=True),
    Row(review="Boring from start to finish. Not a single scene held my attention. Save your money.", human_label_movie_good_bad=False),
    Row(review="Surprisingly good! Went in with low expectations and came out thoroughly entertained.", human_label_movie_good_bad=True),
    Row(review="Overhyped and underdelivered. The trailer was better than the entire movie.", human_label_movie_good_bad=False),
]

reviews_df = spark.createDataFrame(fake_data)

reviews_df.write.mode("overwrite").saveAsTable("classic_stable_been2c_catalog.demos.movie_reviews")

display(reviews_df)

### Option 1: Store Base Prompt, Model, Version in a Delta Table

**Table containing base prompt, model, and prompt + model version**

In [0]:
%sql
CREATE OR REPLACE TABLE classic_stable_been2c_catalog.demos.movie_review_ai_inputs
AS
SELECT
  'Classify the movie review as positive (True) or negative (False). Movie review: ' AS prompt,
  'databricks-claude-opus-4-6' AS model,
  '0' AS version;

SELECT * FROM classic_stable_been2c_catalog.demos.movie_review_ai_inputs;

**Run ai_query against newest base prompt and model**

In [0]:
from pyspark.sql.functions import col

prompt = inputs["prompt"]
model_name = inputs["model"]
review_table_path = 'classic_stable_been2c_catalog.demos.movie_reviews'
prompt_table_path = 'classic_stable_been2c_catalog.demos.movie_review_ai_inputs'

# load in versioned prompt, model by newest version
inputs = spark.table(f"{prompt_table_path}") \
    .orderBy(col("version").cast("int").desc()) \
    .limit(1) \
    .collect()[0]

# define a function that encapsulates the ai_query call
def ai_model_review(model, prompt, table_path):

  # guarantee that results are json
  response_format = '''
  {
    "type": "json_schema",
    "json_schema": {
      "name": "result",
      "schema": {
        "type": "object",
        "properties": {
          "label": {"type": "boolean"}
        }
      },
      "strict": true
    }
  }
  '''

  query = f"""
      SELECT 
        *,
        ai_query(
          '{model_name}',
          CONCAT('{prompt}', review),
          responseFormat => '{response_format}',
          failOnError => false
        ) AS ai_label_movie_good_bad
      FROM {table_path}
  """
  
  df = spark.sql(query)

  return df

df = ai_model_review(model_name, prompt, table_path)

display(df)

### Option 2: Define model and prompt directly in Unity Catalog registered function - version like code in Github

In [0]:
%sql
CREATE OR REPLACE FUNCTION classic_stable_been2c_catalog.demos.fx_ai_label_movie_reviews(review STRING)
COMMENT "Labels a moview review as good or bad"
RETURN FROM_JSON(
  ai_query(
    'databricks-claude-opus-4-6',
    CONCAT('Classify the movie review as positive (True) or negative (False). Movie review: ', review),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "result",
        "schema": {
          "type": "object",
          "properties": {
            "label": {"type": "boolean"}
          }
        },
        "strict": true
      }
    }',
    failOnError => false -- will only fail that row
  ).result,
  'STRUCT<label:BOOLEAN>'
);

In [0]:
df = spark.sql("""
  SELECT 
    *,
    classic_stable_been2c_catalog.demos.fx_ai_label_movie_reviews(review) AS ai_label_movie_good_bad
  FROM classic_stable_been2c_catalog.demos.movie_reviews
  """)

display(df)

### Tracing and Evaluation: Store results in a Delta Table and/or use MLFlow

You can still use MLFlow to log metrics. For tracing overtime, you will need to store in a Delta Table. 

In [0]:
import mlflow
import json
from mlflow.genai.scorers import Correctness

mlflow.set_experiment(experiment_id="859059537351640")

with mlflow.start_run(run_name="ai_label_movie_reviews") as run:

    # Log parameters
    mlflow.log_params({
        "model": "databricks-claude-opus-4-6",
        "uc_function": "classic_stable_been2c_catalog.weather.fx_ai_label_movie_reviews",
        "sample_size": 5,
        "judge": "Correctness"
    })

    # Call the UDF and get human labels in one query
    df = spark.sql("""
    SELECT 
        *,
        classic_stable_been2c_catalog.demos.fx_ai_label_movie_reviews(review) AS ai_label_movie_good_bad
    FROM classic_stable_been2c_catalog.demos.movie_reviews
    """)

    results = df.collect()

    # Use Correctness LLM judge to evaluate each prediction
    correctness_judge = Correctness()
    judge_results = []

    for i, row in enumerate(results):
        feedback = correctness_judge(
            inputs={"request": f"Is the movie review good or bad. Review: {row['review']}"},
            outputs={"response": f"AI Review: {row['ai_label_movie_good_bad']}"},
            expectations={"expected_response": f"Human Review: {row['human_label_movie_good_bad']}"}
        )
        judge_results.append({
            "review": row["review"],
            "human_label": row['human_label_movie_good_bad'],
            "model_prediction": row["ai_label_movie_good_bad"],
            "judge_score": feedback.score if hasattr(feedback, 'score') else str(feedback),
            "judge_rationale": feedback.rationale if hasattr(feedback, 'rationale') else str(feedback)
        })

    # Log judge outputs to MLflow
    scores = [r["judge_score"] for r in judge_results if isinstance(r["judge_score"], (int, float))]
    if scores:
        mlflow.log_metrics({
            "correctness_judge_mean": sum(scores) / len(scores),
            "correctness_judge_min": min(scores),
            "correctness_judge_max": max(scores),
        })

    # Log full judge details as a JSON artifact
    judge_artifact_path = "/tmp/correctness_judge_results.json"
    with open(judge_artifact_path, "w") as f:
        json.dump(judge_results, f, indent=2, default=str)
    mlflow.log_artifact(judge_artifact_path)

    # Manual accuracy metrics
    total = len(results)
    correct = sum(1 for r in results if r["human_label_movie_good_bad"] == r["ai_label_movie_good_bad"])
    accuracy = correct / total if total > 0 else 0

    mlflow.log_metrics({
        "accuracy": accuracy,
        "total_samples": total,
        "correct_predictions": correct,
    })

